In [30]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

In [31]:
# Import Original Files
H_P1 = pd.read_csv("Data//Original_Data//HarrisPartI.csv")          
H_P2 = pd.read_csv("Data//Original_Data//HarrisPartII.csv")         
H_P3 = pd.read_csv("Data//Original_Data//HarrisPartIII.csv")        
V_T2 = pd.read_csv("Data//Original_Data//vandenBerg_table2.csv")        
K_21 = pd.read_csv("Data//Original_Data//Krause21.csv")             

# Imported DataFrames are combined into a "DataList" (List of DataFrames) to allow for iteration later. 
DL=[H_P1,H_P2,H_P3,V_T2,K_21]

## Part1: Standardise IDs

Broadly speaking, each dataset will have a primary "ID" for each object (such as an NGC) and a secondary "Name" that can be used as a backup. Many objects are primarily reffered to by an NGC, for example "NGC 1234". Different data sources rarely agree on naming conventions, for example:
- in Harris data, NGC IDs are formatted as "NGC 1234"
- in Krause, the format is "NGC1234"
- in vandenBerg, there is a specific column only for the NGC number string: "1234"

Hence if we want to use data from multiple sources, we first need to standardise the naming system. Detailed below is the naming system that will be used for this project.
- Each object will have a primary "ID" and a secondary "Name"
- Each object must have an primary "ID". Objects are not required to have a secondary "Name". Objects cannot have the same primary "ID" and secondary "Name"
- Seconary Names are only for helping to resolve cases where two datasets have different primary names for an object and will be dropped once completed.

Rules for assigning IDs:
1) Use NGC if available.
2) If no NGC use the object's primary ID name from the dataset the object is imported from.
3) If no available primary ID name, use the object's secondary name.

Rules for assigning Names:
1) Use the the secondary name from the dataset the object is imported from.
2) Delete the secondary name if it has been substituted in for a missing primary ID.

Formatting Rules:
- No Whitespaces in primary IDs or secondary Names. Eg: "NGC1234", "Pal12"

In [32]:
#Import our raw data

# DL[3] is the vandenBerg data. Here we need to convert the NGC number into an ID.
DL[3].loc[DL[3]["#NGC"]!="XXXX","#NGC"] = "NGC" + DL[3].loc[DL[3]["#NGC"]!="XXXX","#NGC"]   # If the #NGC number exists (eg:1234), we change it to the format "NGC1234"
DL[3].loc[DL[3]["#NGC"]=="XXXX","#NGC"] =         DL[3].loc[DL[3]["#NGC"]=="XXXX","Name"]   # If the #NGC number dosent exist, we sub in its "Name"
DL[3].loc[DL[3]["#NGC"]==DL[3]["Name"],"Name"] =  "XXXX"                                    # If we subbed in the "Name" for a missing NGC, we cut out the name to avoid double counting later.
DL[3].rename(columns={'#NGC': 'ID'}, inplace=True)                                          # Renames the column that now stores our IDs

# DL[4] is the Krause data. Here we just need to rename some columns.
DL[4].rename(columns={'Object': 'ID'}, inplace=True)
DL[4].rename(columns={'AltName': 'Name'}, inplace=True)

## Part2: Correct for different spellings



In [33]:
def NamesAndIDs(DL): #Function for getting all object IDs in the DataList
    AllIDs = []
    AllNames = []
    for i in range(len(DL)):
        DL[i]["ID"] = DL[i]["ID"].str.replace(" ","")
        AllIDs += (DL[i]["ID"].tolist())
        if "Name" in DL[i].columns:
            DL[i]["Name"] = DL[i]["Name"].str.replace(" ","")
            AllNames += DL[i]["Name"].tolist()
    #print(np.unique(AllIDs))
    UniqueNames = np.unique(AllNames)
    #print(UniqueNames)
    return AllIDs, UniqueNames

def FindOrphanIDs(ID_List): #Function for finding which IDs are only used once (These are likely to be objects that are already in the data but under a differently formatted ID)
    UniqueIDs,Counts = np.unique(ID_List,return_counts=True)
    Orphans = UniqueIDs[Counts==1]
    return Orphans

def FindSimilarIDs(ID_List, Prefix=None, Suffix=None): #Function to help find IDs that may refer to the same object but with differently formatted IDs
    SimilarPrefix = []
    SimilarSuffix = []
    PreLen=len(str(Prefix))
    SufLen=len(str(Suffix))
    for ID in ID_List:
        if (Prefix == None) or (ID[:PreLen] == str(Prefix)):
            SimilarPrefix += [ID]
        if (Suffix == None) or (ID[-SufLen:] == str(Suffix)):
            SimilarSuffix += [ID]
    return np.intersect1d(SimilarSuffix,SimilarPrefix)

def AutoOrphanSolver(ID_List):
    for ID in FindOrphanIDs(ID_List):
        Similars = FindSimilarIDs(ID_List, Prefix=str(ID[0]), Suffix=str(ID[-1]))       
        print("Orphan: "+str(ID))
        print("Similar:"+str(Similars))
        print("----------")

AllIDs, UniqueNames = NamesAndIDs(DL)

Looking at all the object IDs:

In [34]:
print(np.unique(AllIDs))

['1636-283' '2MS-GC01' '2MS-GC02' 'AM1' 'AM4' 'Arp2' 'Arp21' 'BH176'
 'BH261' 'Djorg1' 'Djorg2' 'E3' 'ESO-SC06' 'Eridanus' 'FSR1735'
 'GLIMPSE01' 'GLIMPSE02' 'HP1' 'IC1257' 'IC1276' 'IC4499' 'Ko1' 'Ko2'
 'Liller1' 'Lynga7' 'NGC104' 'NGC1261' 'NGC1851' 'NGC1904' 'NGC2298'
 'NGC2419' 'NGC2808' 'NGC288' 'NGC3201' 'NGC362' 'NGC4147' 'NGC4372'
 'NGC4590' 'NGC4833' 'NGC5024' 'NGC5053' 'NGC5139' 'NGC5272' 'NGC5286'
 'NGC5466' 'NGC5634' 'NGC5694' 'NGC5824' 'NGC5897' 'NGC5904' 'NGC5927'
 'NGC5946' 'NGC5986' 'NGC6093' 'NGC6101' 'NGC6121' 'NGC6139' 'NGC6144'
 'NGC6171' 'NGC6205' 'NGC6218' 'NGC6229' 'NGC6235' 'NGC6254' 'NGC6256'
 'NGC6266' 'NGC6273' 'NGC6284' 'NGC6287' 'NGC6293' 'NGC6304' 'NGC6316'
 'NGC6325' 'NGC6333' 'NGC6341' 'NGC6342' 'NGC6352' 'NGC6355' 'NGC6356'
 'NGC6362' 'NGC6366' 'NGC6380' 'NGC6388' 'NGC6397' 'NGC6401' 'NGC6402'
 'NGC6426' 'NGC6440' 'NGC6441' 'NGC6453' 'NGC6496' 'NGC6517' 'NGC6522'
 'NGC6528' 'NGC6535' 'NGC6539' 'NGC6540' 'NGC6541' 'NGC6544' 'NGC6553'
 'NGC6558' 'NGC6569'

Looking at all the IDs that are only used once:

In [35]:
print(FindOrphanIDs(AllIDs))

['Arp21' 'Palomar12' 'Ruprecht106' 'Ter8']


Finding simiar IDs to check for different spellings of the same object names

In [36]:
AutoOrphanSolver(AllIDs)

Orphan: Arp21
Similar:['AM1' 'Arp21']
----------
Orphan: Palomar12
Similar:['Pal12' 'Pal2' 'Palomar12']
----------
Orphan: Ruprecht106
Similar:['Rup106' 'Ruprecht106']
----------
Orphan: Ter8
Similar:['Ter8' 'Terzan8']
----------


In [37]:
print(FindSimilarIDs(UniqueNames, Prefix="A", Suffix="21"))

[]


In [38]:
ID_Aliases = {
    "Palomar12"     :   "Pal12",
    "Ruprecht106"   :   "Rup106",
    "Ter8"          : "Terzan8" }

In [39]:
for i in range(len(DL)):
    DL[i]["ID"]=DL[i]["ID"].replace(ID_Aliases)

In [40]:
AllIDs, UniqueNames = NamesAndIDs(DL)
print(FindOrphanIDs(AllIDs))

['Arp21']


In [41]:
# Thingy to save to Directory:
DirName = "Standardised_IDs"
if not os.path.exists("Data//%s"%DirName):   #Check if the folder has been created
    os.makedirs("Data//%s"%DirName)

FileNames = [
    "HarrisPartI",
    "HarrisPartII",
    "HarrisPartIII",
    "vandenBerg_table2",
    "Krause21"
]

for i in range(len(DL)):
    path = "Data//"+str(DirName)+"//"+str(FileNames[i])+".csv"
    if os.path.exists(path):
        yn = input("The file '%s' already exists. Overwrite [y/n]?"%FileNames[i])
        if yn == "y":
            DL[i].to_csv(path)
    else:   DL[i].to_csv(path)